# Secure credentials with Bedrock AgentCore Identity

This workshop demonstrates how to integrate Strands Agents with Amazon Bedrock AgentCore Identity to securely manage API keys and credentials for external services when building AI agents.

## Overview

In this lab, you will:
- Learn about secure credential management challenges
- Understand Bedrock AgentCore Identity capabilities
- Create API Key Credential Providers for external services
- Test secure credential retrieval in AI agents
- Explore best practices for credential management

## Prerequisites

Before starting this lab, ensure you have:
- AWS credentials configured (IAM role or environment variables)
- Required Python packages installed
- Nova Pro model ID based on AWS region
- An external API key (e.g., Exa API key) for testing

If you're not running in an environment with an IAM role assumed, set your AWS credentials as environment variables:

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

Install required packages for Strands Agents and Bedrock AgentCore Python SDK:

In [1]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

Setup Nova Pro model ID based on AWS region:

In [2]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Nova Pro Model ID: us.amazon.nova-pro-v1:0


## Retrieve Exa API key to connect Remote Exa MCP

In this lab, we will interact [Remote Exa MCP](https://docs.exa.ai/reference/exa-mcp) to perform real-time web searches through the Exa Search API, which requires Exa API key to connect.

Exa MCP server URL: ```https://mcp.exa.ai/mcp?exaApiKey=your-exa-api-key```

To get Exa API key, go to [Exa login page](https://dashboard.exa.ai/login) to register with your email. 

Then go to [API key section](https://dashboard.exa.ai/api-keys) in Exa dashboard to create an API key. Copy the API key to `EXA_API_KEY` in below code...

## Understanding Credential Management Challenges

### Demonstrating Insecure API Key Usage

Let's first demonstrate what happens when we try to use an external service (Exa search) without proper credential management. This will show the security risks and authentication failures that occur with hardcoded or invalid API keys.

⚠️ **Important**: Replace the placeholder API key with your actual Exa API key.

In [3]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

# !-------- UPDATE THE EXA API KEY HERE  --------!
EXA_API_KEY = "{KEY}" 

# Connect to the weather MCP server
print("\nConnecting to MCP Server...")
exa_server = MCPClient(lambda: streamablehttp_client(f"https://mcp.exa.ai/mcp?exaApiKey={EXA_API_KEY}"))

with exa_server:
    # Combine all tools - they all work the same way!
    mcp_tools = (
        exa_server.list_tools_sync()
    )

    print(f"Available tools: {[tool.tool_name for tool in mcp_tools]}")
    
    # Create agent with Exa MCP tools
    agent = Agent(model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
                  system_prompt="You are a helpful assistant that provides concise responses.",
                  tools=mcp_tools)

    agent("What is Bedrock AgentCore?")


Connecting to MCP Server...
Available tools: ['web_search_exa', 'get_code_context_exa']
<thinking> To answer the user's question about Bedrock AgentCore, I need to gather information about it. I will use the web_search_exa tool to find relevant information about Bedrock AgentCore. </thinking>

Tool #1: web_search_exa
<thinking> The web_search_exa tool encountered an error (401) while trying to fetch information about Bedrock AgentCore. This typically indicates an authorization issue. I will attempt to search again, ensuring the query is correctly formatted. </thinking> 
Tool #2: web_search_exa
<thinking> The web_search_exa tool is consistently returning a 401 error, indicating an authorization issue. Since I cannot resolve this issue, I will inform the user that I am unable to retrieve information about Bedrock AgentCore at this time. </thinking>
I'm sorry, but I'm unable to retrieve information about Bedrock AgentCore due to an authorization error. Please try again later or check if 

## What is Bedrock AgentCore Identity?

Amazon Bedrock AgentCore Identity provides secure credential management for AI agents that need to access external services. Key benefits include:

- **Secure Storage**: Stores API keys, tokens, and credentials in AWS Secrets Manager
- **Runtime Retrieval**: Provides secure credential access at runtime without hardcoding
- **Access Control**: Integrates with AWS IAM for fine-grained access permissions
- **Audit Trail**: Maintains logs of credential access for security monitoring
- **Rotation Support**: Enables automatic credential rotation and lifecycle management

This service eliminates the need to hardcode sensitive credentials in your application code, reducing security risks and improving compliance.

## Creating Secure Credential Management

### Step 1: Create API Key Credential Provider

Now we'll create an API Key Credential Provider using Bedrock AgentCore Identity. This securely stores the Exa API key in AWS Secrets Manager and provides a managed way to access it without exposing credentials in code.

⚠️ **Important**: Replace the placeholder API key with your actual Exa API key.

In [5]:
from bedrock_agentcore.services.identity import IdentityClient
from botocore.exceptions import ClientError
import boto3

# !-------- UPDATE THE EXA API KEY HERE  --------!
EXA_API_KEY = "{KEY}" 

region = boto3.session.Session().region_name

#Configure API Key Provider
identity_client = IdentityClient(region=region)

try:
    api_key_provider = identity_client.create_api_key_credential_provider({
        "name": "exa-apikey-provider",
        "apiKey": EXA_API_KEY # Replace it with the API key you obtain from the external application vendor, e.g., OpenAI
    })
    print("Created AgentCore Identity API Key Credential Provider.")
    print(api_key_provider)
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        print("AgentCore Identity API Key Credential Provider already exist.")
    else:
        print(f"ERROR: {e}")
except Exception as e:
    # Show any errors during api key provider creation
    print(f"ERROR: {e}")

Created AgentCore Identity API Key Credential Provider.
{'ResponseMetadata': {'RequestId': '86233ee2-bc58-4372-8ec1-73107e172f12', 'HTTPStatusCode': 201, 'HTTPHeaders': {'date': 'Sun, 15 Mar 2026 09:16:52 GMT', 'content-type': 'application/json', 'content-length': '326', 'connection': 'keep-alive', 'x-amzn-requestid': '86233ee2-bc58-4372-8ec1-73107e172f12', 'x-amzn-remapped-x-amzn-requestid': 'efaa7ee9-3dd3-4b7b-8f50-f565c5e461dd', 'x-amzn-remapped-content-length': '326', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'aQfYuGhtvHcEpoQ=', 'x-amzn-trace-id': 'Root=1-69b67904-4b6f5a1876ffa6c9356ece3f', 'x-amzn-remapped-date': 'Sun, 15 Mar 2026 09:16:52 GMT'}, 'RetryAttempts': 0}, 'apiKeySecretArn': {'secretArn': 'arn:aws:secretsmanager:us-west-2:913429364233:secret:bedrock-agentcore-identity!default/apikey/exa-apikey-provider-jOmg8v'}, 'name': 'exa-apikey-provider', 'credentialProviderArn': 'arn:aws:bedrock-agentcore:us-west-2:913429364233:token-vault/default/apikeycredenti

### Step 2: Test Secure Credential Retrieval

Now let's test our agent using secure credential retrieval. The `@requires_api_key` decorator automatically retrieves the API key from the credential provider at runtime, ensuring no hardcoded secrets in our code while maintaining security best practices.

In [8]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.identity.auth import requires_api_key

@requires_api_key(provider_name="exa-apikey-provider")
def need_api_key(*, api_key: str):
    print(f'received api key for async func: {api_key}')
    return api_key

EXA_API_KEY = need_api_key()

# Connect to the EXA MCP server
print("\nConnecting to MCP Server...")
exa_server = MCPClient(lambda: streamablehttp_client(f"https://mcp.exa.ai/mcp?exaApiKey={EXA_API_KEY}"))

with exa_server:
    # Combine all tools - they all work the same way!
    mcp_tools = (
        exa_server.list_tools_sync()
    )

    print(f"Available tools: {[tool.tool_name for tool in mcp_tools]}")
    
    # Create agent with Exa MCP tools
    agent = Agent(model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
                  system_prompt="You are a helpful assistant that provides concise responses.",
                  tools=mcp_tools)

    agent("What is Bedrock AgentCore?")

Found existing workload identity from /workshop/03-bedrock-agentcore-identity-apikey/.agentcore.json: workload-ID
Found existing user id from /workshop/03-bedrock-agentcore-identity-apikey/.agentcore.json: {USER-ID}


received api key for async func: {KEY}

Connecting to MCP Server...
Available tools: ['web_search_exa', 'get_code_context_exa']
<thinking> To answer the user's question about Bedrock AgentCore, I need to gather information about it. I will use the web_search_exa tool to search for details on Bedrock AgentCore. </thinking>

Tool #1: web_search_exa
<thinking> The web_search_exa tool encountered a 401 error, which typically indicates an authentication issue. Since I can't resolve this directly, I'll attempt to find relevant information using an alternative approach. I'll use the get_code_context_exa tool to search for information related to Bedrock AgentCore on GitHub, Stack Overflow, and official documentation. </thinking> 
Tool #2: get_code_context_exa
<thinking> Both tools have encountered authentication issues, which means I can't retrieve external information about Bedrock AgentCore. Given this limitation, I'll provide a general response based on my existing knowledge. </thinking>

B

Let's examine the detailed execution flow of the agent loop to understand how the agent processes requests and generates responses:

In [9]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 3

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                    ┃ Tool Name            ┃ Tool Input              ┃ Tool Result              ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ What is Bedrock         │                      │                         │                          │
│           │ AgentCore?              │                      │                         │                          │
├───────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼──────────────────────────┤
│ assistant │ <thinking> To answer    │ web_search_exa       │ {                       │                          │
│           │ the user's question     │                      │   "livecrawl":          │                          │
│           │ about Bedrock           │                      │ "fallback",             │                          │
│           │ AgentCore, I need to    │                      │   "query": "Bedrock     │                          │
│           │ gather information      │                      │ AgentCore",             │                          │
│           │ about it. I will use    │                      │   "numResults": 5       │                          │
│           │ the web_search_exa tool │                      │ }                       │                          │
│           │ to search for details   │                      │                         │                          │
│           │ on Bedrock AgentCore.   │                      │                         │                          │
│           │ </thinking>             │                      │                         │                          │
│           │                         │                      │                         │                          │
├───────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼──────────────────────────┤
│ user      │                         │                      │                         │ {                        │
│           │                         │                      │                         │   "text": "Search error  │
│           │                         │                      │                         │ (401): Request failed    │
│           │                         │                      │                         │ with status code 401"    │
│           │                         │                      │                         │ }                        │
├───────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼──────────────────────────┤
│ assistant │ <thinking> The          │ get_code_context_exa │ {                       │                          │
│           │ web_search_exa tool     │                      │   "tokensNum": 5000,    │                          │
│           │ encountered a 401       │                      │   "query": "Bedrock     │                          │
│           │ error, which typically  │                      │ AgentCore"              │                          │
│           │ indicates an            │                      │ }                       │                          │
│           │ authentication issue.   │                      │                         │                          │
│           │ Since I can't resolve   │                      │                         │                          │
│           │ this directly, I'll     │                      │                         │                          │
│           │ attempt to find         │                      │                         │                          │
│           │ relevant information    │                 

## Resource Cleanup (Optional)

Clean up the deployed resources:

In [10]:
import boto3

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)

try:
    print("Deleting AgentCore Identity...")
    agentcore_control_client.delete_api_key_credential_provider(name="exa-apikey-provider")
    print("✓ AgentCore Identity deletion initiated")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

Deleting AgentCore Identity...
✓ AgentCore Identity deletion initiated


## Conclusion

In this lab, you successfully:

- ✅ Identified security risks of hardcoded API keys in agent applications
- ✅ Created a secure API Key Credential Provider using Bedrock AgentCore Identity
- ✅ Implemented secure credential retrieval for external service integration
- ✅ Tested secure credential access with Exa API for web search functionality

## Key Benefits of AgentCore Identity

- **Secure Credential Storage**: Encrypted storage of API keys and secrets
- **Access Control**: Fine-grained permissions for credential access
- **Audit Trail**: Complete logging of credential usage and access
- **Integration Ready**: Seamless integration with MCP servers and agent workflows
- **Best Practices**: Eliminates hardcoded credentials and security vulnerabilities
